# 02 — Clustering Pipeline
## MIT Sloan SSAC 2026 Hackathon

**Mission:** Take the clean data, find natural fan segments, and validate
that they're not just statistical hallucinations.

This is the "putting people in boxes" phase. Except the boxes are
statistically derived, the people are anonymous fan records, and
the whole thing costs $0 in therapy.

---

## 1. Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../scripts'))
from preprocessing import *
from clustering import *
from profiling import *
from visualization import *

%matplotlib inline
print('🚀 All modules loaded. Clustering engines standing by.')

In [ ]:
# Load and clean data (re-run from 01_eda or load fresh)
# ============================================================
# ⬆️⬆️⬆️  CHANGE THE FILENAME BELOW  ⬆️⬆️⬆️
# ============================================================
df = load_and_inspect('../data/CHANGEME.csv')
col_types = identify_column_types(df)
df = clean_data(df, col_types)

## 2. Feature Selection

Auto-select features or manually pick the ones that matter most.
The auto-selector drops IDs, low-variance, and redundant columns.

**Pro tip:** If you know which features should drive segmentation,
use the manual override. Domain knowledge > algorithms.

In [ ]:
# Auto feature selection — let the algorithm play Marie Kondo
feature_df = create_feature_matrix(df)
print(f'\nSelected features: {feature_df.columns.tolist()}')

In [ ]:
# ======================================
# MANUAL OVERRIDE — uncomment to pick your own features
# ======================================
# Sometimes you know better than the algorithm. That's okay.
# Just list the columns that should drive your segmentation.
#
# features = ['col_a', 'col_b', 'col_c', 'col_d']  # CHANGE THESE
# feature_df = df[features]
# print(f'Using {len(features)} manually selected features')

## 3. Encoding

Transform human-readable data into the numeric matrix that clustering
algorithms require. Scales numeric, encodes categorical, drops the rest.

In [ ]:
# Re-detect types on the feature subset
feature_col_types = identify_column_types(feature_df)

X, scaler, encoders, feature_names, cat_idx = encode_for_clustering(feature_df, feature_col_types)
print(f'Feature matrix shape: {X.shape}')
print(f'Feature names: {feature_names}')

## 4. Find Optimal K

Test multiple values of k and look at the evidence. The algorithm will
recommend one, but the elbow plot and silhouette curve are your real
advisors. Use your eyes. Use your brain. Use your gut if necessary.

In [ ]:
k_results = find_optimal_k(X, categorical_indices=cat_idx if cat_idx else None)

# Display the scores table
display(k_results['scores'])

In [ ]:
# ======================================
# MANUAL OVERRIDE — uncomment to pick your own k
# ======================================
# If you disagree with the recommendation (it happens),
# set k manually here.
#
# k = 4  # Override with your chosen k
# print(f'Using manual k = {k}')

# Or use the recommendation
k = k_results['recommended_k']
print(f'Using recommended k = {k}')

## 5. Run Clustering

Two options:
- **auto_cluster:** Picks the best method based on your data type
- **Manual:** Call a specific algorithm yourself

Start with auto, then switch to manual if you want more control.

In [ ]:
# Option A: Auto cluster (recommended for first pass)
labels, k, method, meta = auto_cluster(X, col_types, cat_idx if cat_idx else None)
print(f'\n📋 Method: {method}, k = {k}')

In [ ]:
# ======================================
# MANUAL ALTERNATIVES — uncomment one
# ======================================
#
# Option B: Explicit K-Means
# labels = run_kmeans(X, k=4)
#
# Option C: K-Prototypes (mixed data)
# labels = run_kprototypes(X, k=4, categorical_indices=cat_idx)
#
# Option D: Hierarchical
# labels = run_hierarchical(X, k=4)
#
# Option E: HDBSCAN (auto k)
# labels = run_hdbscan_clustering(X)
#
# Option F: K-Modes (all categorical)
# labels = run_kmodes(X, k=4)

## 6. Stability Check

Are these clusters real or just artifacts of a random seed?
Run it 10 times and see if we get the same answer. Like asking
"are you sure?" but mathematically.

In [ ]:
stability = cluster_stability_check(X, k, categorical_indices=cat_idx if cat_idx else None)

## 7. Dimensionality Reduction + Scatter Plot

Squish your high-dimensional data into 2D so humans can see it.
UMAP > PCA for visualizing cluster separation (usually).

In [ ]:
X_2d = reduce_dimensions(X, method='umap')

# Plot with temporary names (we'll name them properly in step 9)
temp_names = {i: f'Cluster {i}' for i in range(k)}
fig = plot_cluster_scatter(X_2d, labels, temp_names, method_name='UMAP')
fig.show()

## 8. Quick Profile Preview

First look at what makes each cluster different. This feeds into naming.

In [ ]:
profiles = profile_clusters(df, labels, col_types)
display(profiles)

## 9. Name the Clusters

Auto-generated names are a starting point. They're like
baby names picked by an AI — technically valid, occasionally
inspired, often needs human intervention.

In [ ]:
cluster_names = name_clusters(profiles, k)
print('\n📝 Auto-generated cluster names:')
for cid, name in cluster_names.items():
    print(f'   {cid}: {name}')

In [ ]:
# ======================================
# MANUAL OVERRIDE — rename clusters here
# ======================================
# Once you understand the data, give them real names:
#
# cluster_names = {
#     0: 'The Values Advocate',
#     1: 'The Silent Superfan',
#     2: 'The Social Spectator',
#     3: 'The Casual Curious',
# }

# Update the scatter plot with real names
fig = plot_cluster_scatter(X_2d, labels, cluster_names, method_name='UMAP')
fig.show()

## 10. Save Results

Save the clustered data for use in `03_analysis.ipynb`.

In [ ]:
df['cluster'] = labels
df.to_csv('../data/clustered_data.csv', index=False)
print(f'✅ Saved clustered data to data/clustered_data.csv')
print(f'   {len(df):,} rows, {k} clusters')
print(f'   Method: {method}')
print(f'   Stability: {stability:.3f}')
print(f'\n🎯 Proceed to 03_analysis.ipynb for CVI and deep analysis.')